# Notebook 05: Automate staged deployment with a SageMaker Project

In this notebook, you package the editable deployment assets as an organization-owned S3 Project template and launch a SageMaker Project. The Project watches the Model Package Group used in notebooks 01–04 and runs only the deployment path: Model Registry and source events → CodePipeline → staging endpoint → test → manual approval → production endpoint.

|||
|---|---|
|00. Environment setup ||
|01. Experiment in a notebook, then scale with SageMaker AI jobs ||
|02. Automate model governance with EventBridge and Model Cards ||
|03. Operationalize with an ML pipeline and model registry ||
|04. Deploy to an endpoint and monitor ||
|**05. Templatize staged deployment as a Project** |**◄ YOU ARE HERE**|

## What you do here

1. Package [`deployment-seed-code/`](deployment-seed-code/) into a ZIP.
2. Upload the ZIP and [`cfn/deployment_only_project_template.yml`](cfn/deployment_only_project_template.yml) to your S3 bucket.
3. Tag the template for SageMaker Studio visibility.
4. Create a SageMaker Project with `TemplateProviders/CfnTemplateProvider`.
5. Follow the resulting CodePipeline as it deploys to staging, runs CSV inference tests, waits for guarded manual approval, then deploys and tests production.

**Outcome:** an organization-owned deployment-only Project template that deploys an approved model to staging, tests it, and gates production behind manual approval.

> **Support caveat:** this minimal derivative retains CodeCommit because it matches the copied AWS stack and this workshop account has existing access. AWS labels the original product not supported, and CodeCommit is unavailable to new customers. Use the newer CodeConnections-based `deploy-codepipeline.yaml` sample for a new production platform.

> **Cost warning:** publishing two small S3 objects is inexpensive. Creating the Project provisions CodeCommit, CodeBuild, CodePipeline, EventBridge, CloudFormation, and S3 resources. A deployment pipeline run creates a billable staging endpoint and, after approval, a production endpoint. All mutating actions are disabled by default.

## Architecture

![SageMaker Project deployment toolchain with Model Registry and source triggers, staged deployment, testing, manual approval, and production](img/deployment-project-architecture-aws.svg)

The diagram separates the two event triggers from the deployment workflow. The generated Project owns the source repository, EventBridge rules, artifact bucket, build and test projects, and CodePipeline. The staging and production CloudFormation actions each create a SageMaker Model, an EndpointConfig with data capture, and an Endpoint.


## 1. Runtime and AWS context

Run this notebook from the repository root with the workshop's Python 3.12 kernel. It reuses `%store` values when available but does not depend on the model-build notebook.

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}"
)

try:
    get_ipython().run_line_magic("store", "-r")
except Exception:
    pass

In [ ]:
import hashlib
import json
import re
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.parse import quote

import boto3
import yaml
from botocore.exceptions import ClientError

session = boto3.Session()
region = globals().get("region") or session.region_name
if not region:
    raise RuntimeError("No AWS Region is configured.")

sts = session.client("sts", region_name=region)
account_id = sts.get_caller_identity()["Account"]
s3 = session.client("s3", region_name=region)
sm = session.client("sagemaker", region_name=region)
cfn = session.client("cloudformation", region_name=region)
iam = session.client("iam", region_name=region)

default_bucket = globals().get("bucket_name") or f"sagemaker-{region}-{account_id}"
print(f"Region:  {region}")
print(f"Account: {account_id}")
print(f"Default publication bucket: {default_bucket}")

## 2. Settings and safety controls

The three flags separate publication, optional Studio UI discovery, and Project creation. Programmatic Project creation does not require domain tagging or bucket CORS.

In [ ]:
SEED_DIRECTORY = Path("deployment-seed-code")
PROJECT_TEMPLATE = Path("cfn/deployment_only_project_template.yml")
BUILD_DIRECTORY = Path(".build/deployment-project-template")

PUBLICATION_BUCKET = default_bucket
PUBLICATION_PREFIX = "project-templates/deployment-only"
SEED_OBJECT_KEY = f"{PUBLICATION_PREFIX}/seed/deployment-seed-code.zip"
TEMPLATE_OBJECT_KEY = f"{PUBLICATION_PREFIX}/templates/deployment-only-codecommit.yml"

PUBLISH_ARTIFACTS = True       # creates/updates two S3 objects
CONFIGURE_STUDIO_DISCOVERY = True  # updates bucket CORS and a SageMaker domain tag
CREATE_PROJECT = True          # creates the CI/CD toolchain; later runs can create endpoints

DOMAIN_ID = globals().get("domain_id", "")
date_suffix = time.strftime("%d%m%y", time.gmtime())
PROJECT_NAME = f"aim408-custom-deploy-{date_suffix}"
MODEL_PACKAGE_GROUP_ARN = ""  # optional in preview; required to create a Project
LAUNCH_ROLE_NAME = "AIM408-AmazonSageMakerProjectsLaunchRole"
SERVICE_ROLE_NAMES = [
    "AIM408-AmazonSageMakerProjectsEventsRole",
    "AIM408-AmazonSageMakerProjectsCodeBuildRole",
    "AIM408-AmazonSageMakerProjectsCodePipelineRole",
    "AIM408-AmazonSageMakerProjectsCloudformationRole",
    "AIM408-AmazonSageMakerProjectsExecutionRole",
]

APPROVE_PRODUCTION = True
WAIT_FOR_PRODUCTION_ENDPOINT = True
TEST_PRODUCTION_ENDPOINT = True
# for a full up cleanup
CLEAN_UP_DEPLOYMENT = False
DELETE_RETAINED_ARTIFACT_BUCKET = False

print(f"Seed target:     s3://{PUBLICATION_BUCKET}/{SEED_OBJECT_KEY}")
print(f"Template target: s3://{PUBLICATION_BUCKET}/{TEMPLATE_OBJECT_KEY}")

## 3. Validate and package the editable seed repository

Files are stored at the ZIP root because `AWS::CodeCommit::Repository.Code` expands that root into the new repository. Fixed timestamps and permissions make the archive reproducible: unchanged source produces the same SHA-256. `ORIGIN.md` is included so each generated repository retains provenance.

In [ ]:
required_seed_files = {
    Path("build.py"),
    Path("buildspec.yml"),
    Path("endpoint-config-template.yml"),
    Path("staging-config.json"),
    Path("prod-config.json"),
    Path("test/buildspec.yml"),
    Path("test/test.py"),
    Path("test/payload.csv"),
}

if not SEED_DIRECTORY.is_dir():
    raise FileNotFoundError(SEED_DIRECTORY)
if not PROJECT_TEMPLATE.is_file():
    raise FileNotFoundError(PROJECT_TEMPLATE)

seed_files = sorted(
    path for path in SEED_DIRECTORY.rglob("*")
    if (
        path.is_file()
        and not {"__pycache__", ".ipynb_checkpoints"}.intersection(path.parts)
        and path.suffix != ".pyc"
    )
)
relative_seed_files = {path.relative_to(SEED_DIRECTORY) for path in seed_files}
assert all(".ipynb_checkpoints" not in path.parts for path in relative_seed_files)
missing = required_seed_files - relative_seed_files
if missing:
    raise FileNotFoundError(f"Missing required seed files: {sorted(map(str, missing))}")

compile((SEED_DIRECTORY / "build.py").read_text(), "build.py", "exec")
compile((SEED_DIRECTORY / "test/test.py").read_text(), "test/test.py", "exec")
json.loads((SEED_DIRECTORY / "staging-config.json").read_text())
json.loads((SEED_DIRECTORY / "prod-config.json").read_text())
yaml.safe_load((SEED_DIRECTORY / "buildspec.yml").read_text())
yaml.safe_load((SEED_DIRECTORY / "test/buildspec.yml").read_text())
yaml.compose((SEED_DIRECTORY / "endpoint-config-template.yml").read_text())

print(f"Validated {len(seed_files)} seed files:")
for path in seed_files:
    print(f"  {path.relative_to(SEED_DIRECTORY)}")

In [ ]:
BUILD_DIRECTORY.mkdir(parents=True, exist_ok=True)
seed_zip = BUILD_DIRECTORY / "deployment-seed-code.zip"

with zipfile.ZipFile(seed_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
    for source_path in seed_files:
        relative_path = source_path.relative_to(SEED_DIRECTORY).as_posix()
        info = zipfile.ZipInfo(relative_path, date_time=(2023, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        archive.writestr(info, source_path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)

with zipfile.ZipFile(seed_zip) as archive:
    archived_files = set(archive.namelist())
    expected_files = {path.relative_to(SEED_DIRECTORY).as_posix() for path in seed_files}
    assert archived_files == expected_files
    assert all(not name.startswith("deployment-seed-code/") for name in archived_files)

seed_sha256 = hashlib.sha256(seed_zip.read_bytes()).hexdigest()
print(f"Built {seed_zip} ({seed_zip.stat().st_size:,} bytes)")
print(f"SHA-256: {seed_sha256}")

## 4. Validate the deployment-only project template

The AWS snapshot remains unchanged under `cfn/endpoint_deployment_toolchain_template.yml`. This derivative adds only `DeploymentSeedCodeBucket` and `DeploymentSeedCodeKey`, replacing the hard-coded AWS seed location with references to your uploaded ZIP. SageMaker injects `SageMakerProjectName` and `SageMakerProjectId`; callers provide the model package group and seed location.

In [ ]:
template_bytes = PROJECT_TEMPLATE.read_bytes()
template = yaml.safe_load(template_bytes)
expected_parameters = {
    "SageMakerProjectName",
    "SageMakerProjectId",
    "SourceModelPackageGroupName",
    "DeploymentSeedCodeBucket",
    "DeploymentSeedCodeKey",
}
assert set(template["Parameters"]) == expected_parameters

repository = template["Resources"]["ModelDeployCodeCommitRepository"]
seed_source = repository["Properties"]["Code"]["S3"]
assert seed_source["Bucket"] == {"Ref": "DeploymentSeedCodeBucket"}
assert seed_source["Key"] == {"Ref": "DeploymentSeedCodeKey"}

resource_types = Counter(resource["Type"] for resource in template["Resources"].values())
assert resource_types["AWS::CodeCommit::Repository"] == 1
assert resource_types["AWS::CodePipeline::Pipeline"] == 1
assert resource_types["AWS::CodeBuild::Project"] == 2
assert resource_types["AWS::Events::Rule"] == 2

template_text = template_bytes.decode("utf-8")
assert "ServiceCatalogProducts" not in template_text
for role_name in SERVICE_ROLE_NAMES:
    assert role_name in template_text, f"Template does not reference {role_name}"

template_sha256 = hashlib.sha256(template_bytes).hexdigest()
pipeline = template["Resources"]["ModelDeployPipeline"]
stages = [stage["Name"] for stage in pipeline["Properties"]["Stages"]]
print(f"Template SHA-256: {template_sha256}")
print(f"Parameters: {sorted(expected_parameters)}")
print(f"Pipeline: {' -> '.join(stages)}")

## 5. Check prerequisite roles

The launch role provisions the Project stack. Each AWS service then assumes its own verified `AIM408-AmazonSageMakerProjects*` role: EventBridge starts CodePipeline, CodeBuild renders and tests, CodePipeline orchestrates, CloudFormation deploys endpoint stacks, and SageMaker hosts the model. The preflight also asks IAM whether the launch role may create the `sagemaker-*` CodeCommit repository and pass each service role.

In [ ]:
role_arns = {}
missing_roles = []
for role_name in [LAUNCH_ROLE_NAME, *SERVICE_ROLE_NAMES]:
    try:
        role_arns[role_name] = iam.get_role(RoleName=role_name)["Role"]["Arn"]
        print(f"Found   {role_name}")
    except iam.exceptions.NoSuchEntityException:
        missing_roles.append(role_name)
        print(f"MISSING {role_name}")

permission_failures = []
permission_simulation_available = True
if LAUNCH_ROLE_NAME in role_arns:
    simulations = [
        (
            "codecommit:CreateRepository",
            f"arn:aws:codecommit:{region}:{account_id}:sagemaker-{PROJECT_NAME}-p-example-modeldeploy",
        ),
        *[("iam:PassRole", role_arns[name]) for name in SERVICE_ROLE_NAMES if name in role_arns],
    ]
    for action, resource_arn in simulations:
        try:
            decision = iam.simulate_principal_policy(
                PolicySourceArn=role_arns[LAUNCH_ROLE_NAME],
                ActionNames=[action],
                ResourceArns=[resource_arn],
            )["EvaluationResults"][0]["EvalDecision"]
        except ClientError as error:
            if error.response["Error"]["Code"] in {"AccessDenied", "AccessDeniedException"}:
                permission_simulation_available = False
                print("IAM simulation is unavailable to this caller; role existence was still verified.")
                break
            raise
        print(f"{decision:<14} {action:<32} {resource_arn.rsplit('/', 1)[-1]}")
        if decision != "allowed":
            permission_failures.append((action, resource_arn, decision))

if CREATE_PROJECT:
    if missing_roles:
        raise RuntimeError(f"Cannot create the Project; missing roles: {missing_roles}")
    if permission_failures:
        raise RuntimeError(
            "Launch-role preflight failed. Apply cfn/projects_launch_role_codecommit_policy.md "
            f"and verify PassRole: {permission_failures}"
        )

## 6. Publish the seed ZIP and template to S3

Set `PUBLISH_ARTIFACTS = True` in the settings cell, then rerun from there. Existing objects at these keys are versioned only if bucket versioning is enabled; use a different prefix when you need immutable releases. The template object receives `sagemaker:studio-visibility=true`.

In [ ]:
def ensure_bucket(bucket_name):
    try:
        s3.head_bucket(Bucket=bucket_name)
        return False
    except ClientError as error:
        status = error.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        if status not in {404}:
            raise
    request = {"Bucket": bucket_name}
    if region != "us-east-1":
        request["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**request)
    return True


if not PUBLISH_ARTIFACTS:
    print("PREVIEW ONLY — set PUBLISH_ARTIFACTS = True to upload the artifacts.")
else:
    created = ensure_bucket(PUBLICATION_BUCKET)
    print(f"{'Created' if created else 'Using'} bucket {PUBLICATION_BUCKET}")

    cfn.validate_template(TemplateBody=template_bytes.decode("utf-8"))
    s3.upload_file(
        str(seed_zip), PUBLICATION_BUCKET, SEED_OBJECT_KEY,
        ExtraArgs={"ContentType": "application/zip", "Metadata": {"sha256": seed_sha256}},
    )
    s3.upload_file(
        str(PROJECT_TEMPLATE), PUBLICATION_BUCKET, TEMPLATE_OBJECT_KEY,
        ExtraArgs={"ContentType": "application/x-yaml", "Metadata": {"sha256": template_sha256}},
    )
    s3.put_object_tagging(
        Bucket=PUBLICATION_BUCKET,
        Key=TEMPLATE_OBJECT_KEY,
        Tagging={"TagSet": [{"Key": "sagemaker:studio-visibility", "Value": "true"}]},
    )

    for key, expected_sha in [(SEED_OBJECT_KEY, seed_sha256), (TEMPLATE_OBJECT_KEY, template_sha256)]:
        uploaded = s3.head_object(Bucket=PUBLICATION_BUCKET, Key=key)
        assert uploaded["Metadata"]["sha256"] == expected_sha
        print(f"Published s3://{PUBLICATION_BUCKET}/{key} ({uploaded['ContentLength']:,} bytes)")

## 7. Optionally expose the template in the Studio UI

This is not needed for the programmatic launch below. Studio discovery requires the domain tag `sagemaker:projectS3TemplatesLocation` and bucket CORS. The cell preserves existing CORS rules and adds the required read rule if absent. Set `DOMAIN_ID` explicitly if it was not restored from notebook 00.

In [ ]:
studio_cors_rule = {
    "AllowedHeaders": ["*"],
    "AllowedMethods": ["GET"],
    "AllowedOrigins": ["*"],
    "ExposeHeaders": ["ETag", "x-amz-version-id"],
    "MaxAgeSeconds": 3000,
}

if not CONFIGURE_STUDIO_DISCOVERY:
    print("Skipped — programmatic Project creation does not require Studio discovery.")
else:
    if not PUBLISH_ARTIFACTS:
        raise RuntimeError("Publish the template before configuring Studio discovery.")
    if not DOMAIN_ID:
        domains = sm.list_domains(MaxResults=100)["Domains"]
        if len(domains) != 1:
            raise RuntimeError(f"Set DOMAIN_ID explicitly; found {len(domains)} domains.")
        DOMAIN_ID = domains[0]["DomainId"]

    try:
        cors_rules = s3.get_bucket_cors(Bucket=PUBLICATION_BUCKET)["CORSRules"]
    except ClientError as error:
        if error.response["Error"]["Code"] != "NoSuchCORSConfiguration":
            raise
        cors_rules = []
    if studio_cors_rule not in cors_rules:
        cors_rules.append(studio_cors_rule)
        s3.put_bucket_cors(
            Bucket=PUBLICATION_BUCKET, CORSConfiguration={"CORSRules": cors_rules}
        )

    domain = sm.describe_domain(DomainId=DOMAIN_ID)
    template_location = f"s3://{PUBLICATION_BUCKET}/{PUBLICATION_PREFIX}/templates/"
    sm.add_tags(
        ResourceArn=domain["DomainArn"],
        Tags=[{"Key": "sagemaker:projectS3TemplatesLocation", "Value": template_location}],
    )
    print(f"Configured domain {DOMAIN_ID} to discover {template_location}")

## 8. Select the Model Package Group

Supply a group ARN, not a package-version ARN. If earlier notebooks stored a group name, this cell resolves it automatically. The Project template receives the final group name and its EventBridge rule watches that group.

In [ ]:
from governance.approval_gate import resolve_model_package_group

MODEL_PACKAGE_GROUP_ARN = ""  # e.g. arn:aws:sagemaker:us-west-2:123456789012:model-package-group/my-group

if not MODEL_PACKAGE_GROUP_ARN:
    workshop_group_name = resolve_model_package_group(
        sm, registered_model_name, prefix="aim408-from-idea-to-prod-model-"
    )
    MODEL_PACKAGE_GROUP_ARN = sm.describe_model_package_group(
        ModelPackageGroupName=workshop_group_name
    )["ModelPackageGroupArn"]
    print("No override supplied; using the workshop group.")

print(f"Model Package Group ARN: {MODEL_PACKAGE_GROUP_ARN}")

model_package_group_name = None
if MODEL_PACKAGE_GROUP_ARN:
    match = re.fullmatch(
        r"arn:(?P<partition>aws(?:-[a-z]+)*):sagemaker:(?P<region>[^:]+):"
        r"(?P<account>\d{12}):model-package-group/(?P<name>[A-Za-z0-9](?:-*[A-Za-z0-9]){0,62})",
        MODEL_PACKAGE_GROUP_ARN,
    )
    if not match:
        raise ValueError("MODEL_PACKAGE_GROUP_ARN must be a Model Package Group ARN.")
    if match["region"] != region or match["account"] != account_id:
        raise ValueError("The Model Package Group must be in this account and Region.")
    model_package_group_name = match["name"]
    described = sm.describe_model_package_group(
        ModelPackageGroupName=model_package_group_name
    )
    assert described["ModelPackageGroupArn"] == MODEL_PACKAGE_GROUP_ARN
    print(f"Deployment source: {MODEL_PACKAGE_GROUP_ARN}")
elif CREATE_PROJECT:
    raise RuntimeError("Set MODEL_PACKAGE_GROUP_ARN before creating the Project.")
else:
    print("No group selected — acceptable for artifact publication and launch preview only.")

## 9. Create the deployment-only SageMaker Project

This uses the S3 template directly—there is no custom Service Catalog product to create or discover. SageMaker supplies the project name/ID parameters. The launch call supplies only the Model Package Group and your seed object coordinates. Set both `PUBLISH_ARTIFACTS = True` and `CREATE_PROJECT = True` when ready.

In [ ]:
if not re.fullmatch(r"[A-Za-z](?:-*[A-Za-z0-9]){0,31}", PROJECT_NAME):
    raise ValueError("PROJECT_NAME must start with a letter and contain at most 32 letters, digits, or hyphens.")

template_url = (
    f"https://{PUBLICATION_BUCKET}.s3.{region}.amazonaws.com/"
    f"{quote(TEMPLATE_OBJECT_KEY, safe='/')}"
)
project_details = None
project_created_here = False

existing = [
    item for item in sm.list_projects(NameContains=PROJECT_NAME, MaxResults=100)["ProjectSummaryList"]
    if item["ProjectName"] == PROJECT_NAME
]
if existing:
    project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    print(f"Project already exists: {PROJECT_NAME} ({project_details['ProjectStatus']})")
elif not CREATE_PROJECT:
    print("PREVIEW ONLY — no Project created.")
    print(f"Template URL: {template_url}")
    print(f"SourceModelPackageGroupName: {model_package_group_name or '<set MODEL_PACKAGE_GROUP_ARN>'}")
    print(f"DeploymentSeedCodeBucket: {PUBLICATION_BUCKET}")
    print(f"DeploymentSeedCodeKey: {SEED_OBJECT_KEY}")
else:
    if not PUBLISH_ARTIFACTS:
        raise RuntimeError("Set PUBLISH_ARTIFACTS = True and publish before creating the Project.")
    if missing_roles:
        raise RuntimeError(f"Missing required roles: {missing_roles}")
    response = sm.create_project(
        ProjectName=PROJECT_NAME,
        ProjectDescription=(
            "Deployment-only CI/CD for approved SageMaker Model Packages, published from an S3 custom template."
        ),
        TemplateProviders=[{
            "CfnTemplateProvider": {
                "TemplateName": "DeploymentOnlyCodeCommit",
                "TemplateURL": template_url,
                "RoleARN": role_arns[LAUNCH_ROLE_NAME],
                "Parameters": [
                    {"Key": "SourceModelPackageGroupName", "Value": model_package_group_name},
                    {"Key": "DeploymentSeedCodeBucket", "Value": PUBLICATION_BUCKET},
                    {"Key": "DeploymentSeedCodeKey", "Value": SEED_OBJECT_KEY},
                ],
            }
        }],
        Tags=[
            {"Key": "workshop", "Value": "from-idea-to-production"},
            {"Key": "purpose", "Value": "model-deployment"},
        ],
    )
    project_created_here = True
    project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    print(f"Creating {response['ProjectArn']}")

In [ ]:
if project_details is None:
    print("Skipped — enable CREATE_PROJECT when ready.")
elif not project_created_here:
    print(f"Existing Project status: {project_details['ProjectStatus']}")
else:
    terminal = {"CreateCompleted", "CreateFailed", "CreateCompletedWithErrors"}
    deadline = time.time() + 900
    while project_details["ProjectStatus"] not in terminal:
        if time.time() > deadline:
            raise TimeoutError(f"Project creation did not finish: {project_details['ProjectStatus']}")
        print(f"  {time.strftime('%H:%M:%S')} | {project_details['ProjectStatus']}")
        time.sleep(20)
        project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    if project_details["ProjectStatus"] != "CreateCompleted":
        raise RuntimeError(
            f"Project creation failed: {project_details['ProjectStatus']} — "
            f"{project_details.get('ProjectStatusMessage', '')}"
        )
    print(f"Project ready: {PROJECT_NAME} / {project_details['ProjectId']}")

In [ ]:
from html import escape

from IPython.display import HTML, display

project_url = None
if project_details is None:
    print("No Project is available yet. Create the Project before opening it in Studio.")
else:
    project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    if project_details["ProjectStatus"] != "CreateCompleted":
        print(
            f"Project {PROJECT_NAME} is {project_details['ProjectStatus']}; "
            "the Studio link will be shown after creation completes."
        )
    else:
        project_domain_id = (
            project_details.get("CreatedBy", {}).get("DomainId")
            or DOMAIN_ID
            or globals().get("domain_id")
        )
        if not project_domain_id:
            print("Set DOMAIN_ID to render the SageMaker Studio Project link.")
        else:
            project_name = project_details["ProjectName"]
            project_url = (
                f"https://studio-{project_domain_id}.studio.{region}.sagemaker.aws/"
                f"projects/{quote(project_name, safe='')}/"
            )
            display(HTML(
                f'<p><b>SageMaker Project:</b> <code>{escape(project_name)}</code><br>'
                f'<a target="_blank" rel="noopener noreferrer" href="{escape(project_url)}">'
                "Open the Project in SageMaker Studio</a></p>"
            ))


## Open the deployment resources

The Project API records the CloudFormation stack that provisioned the toolchain. Use that stack to discover the physical CodePipeline name instead of duplicating its naming convention. Then read the deployed pipeline configuration to discover the staging stack and endpoint. These cells are read-only: they create links and report current status without starting or approving a deployment.


In [ ]:
from html import escape

from IPython.display import HTML, display

codepipeline = session.client("codepipeline", region_name=region)
pipeline_name = None
pipeline_state = None

if project_details is None:
    print("No Project is available yet. Create the Project before opening its pipeline.")
else:
    # Refresh the response because it contains the generated Project ID, creator domain,
    # and the physical CloudFormation stack created by the CfnTemplateProvider.
    project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    project_id = project_details["ProjectId"]

    provider = next(
        (item.get("CfnTemplateProviderDetail") for item in project_details.get("TemplateProviderDetails", [])) ,
        None,
    )
    toolchain_stack_name = (provider or {}).get("StackDetail", {}).get("Name")

    if toolchain_stack_name:
        try:
            summaries = cfn.list_stack_resources(StackName=toolchain_stack_name)["StackResourceSummaries"]
            pipeline_resource = next(
                (item for item in summaries if item["LogicalResourceId"] == "ModelDeployPipeline"),
                None,
            )
            if pipeline_resource and pipeline_resource.get("PhysicalResourceId"):
                pipeline_name = pipeline_resource["PhysicalResourceId"]
        except ClientError:
            # The fallback below remains valid while CloudFormation resource metadata catches up.
            pass

    # Fallback for old Project API responses that do not expose StackDetail. The name is
    # part of this template contract and is verified by get_pipeline_state below.
    if not pipeline_name:
        pipeline_name = f"sagemaker-{PROJECT_NAME}-{project_id}-modeldeploy"

    try:
        pipeline_state = codepipeline.get_pipeline_state(name=pipeline_name)
        stage_status = " · ".join(
            f"{stage['stageName']}: {stage.get('latestExecution', {}).get('status', 'Not started')}"
            for stage in pipeline_state["stageStates"]
        )
    except ClientError as error:
        stage_status = f"Pipeline status is not available yet: {error.response['Error']['Code']}"

    pipeline_url = (
        "https://console.aws.amazon.com/codesuite/codepipeline/pipelines/"
        f"{quote(pipeline_name, safe='')}/view?region={quote(region, safe='')}"
    )
    display(HTML(
        f'<p><b>Deployment pipeline:</b> <code>{escape(pipeline_name)}</code><br>'
        f'{escape(stage_status)}<br>'
        f'<a target="_blank" rel="noopener noreferrer" href="{escape(pipeline_url)}">'
        "Open the deployment pipeline in the AWS console</a></p>"
    ))


### Runtime view: CodePipeline execution

The architecture diagram above explains the logical flow. This console view shows an execution after staging deployment and testing, waiting at the manual approval action.

![CodePipeline execution waiting for production approval](img/project-codepipeline.png)


In [ ]:
codecommit = session.client("codecommit", region_name=region)
repository_name = None
repository_url = None

if project_details is None:
    print("No Project is available yet.")
else:
    if toolchain_stack_name:
        try:
            stack_resources = cfn.list_stack_resources(
                StackName=toolchain_stack_name
            )["StackResourceSummaries"]
            repository_resource = next(
                (item for item in stack_resources if item["LogicalResourceId"] == "ModelDeployCodeCommitRepository"),
                None,
            )
            if repository_resource and repository_resource.get("PhysicalResourceId"):
                repository_physical_id = repository_resource["PhysicalResourceId"]
                # CloudFormation reports CodeCommit's repository UUID as the physical ID.
                paginator = codecommit.get_paginator("list_repositories")
                repository_name = next(
                    (
                        item["repositoryName"]
                        for page in paginator.paginate()
                        for item in page["repositories"]
                        if item["repositoryId"] == repository_physical_id
                    ),
                    None,
                )
        except ClientError:
            pass
    if not repository_name:
        repository_name = f"sagemaker-{PROJECT_NAME}-{project_id}-modeldeploy"

    repository = codecommit.get_repository(
        repositoryName=repository_name
    )["repositoryMetadata"]
    repository_url = (
        f"https://console.aws.amazon.com/codecommit/home?region={quote(region, safe='')}"
        f"#/repository/{quote(repository_name, safe='')}/browse/main/--/"
    )
    display(HTML(
        f'<p><b>Deployment source repository:</b> <code>{escape(repository_name)}</code><br>'
        f'<a target="_blank" rel="noopener noreferrer" href="{escape(repository_url)}">'
        "Open the deployment seed in CodeCommit</a><br>"
        f'<small>Clone URL: <code>{escape(repository["cloneUrlHttp"])}</code></small></p>'
    ))

In [ ]:
# Wait only for the endpoint to become visible, not for it to become InService.
# Link rendering never fails on timeout; the page becomes useful as deployment progresses.
WAIT_FOR_STAGING_ENDPOINT_SECONDS = 300
endpoint_name = None
endpoint_status = "Not visible yet"

if project_details is None or not pipeline_name:
    print("No deployment pipeline is available yet.")
else:
    staging_stack_name = None
    try:
        pipeline_definition = codepipeline.get_pipeline(name=pipeline_name)["pipeline"]
        staging_action = next(
            action
            for stage in pipeline_definition["stages"] if stage["name"] == "DeployStaging"
            for action in stage["actions"] if action["name"] == "DeployResourcesStaging"
        )
        staging_stack_name = staging_action["configuration"]["StackName"]
    except (ClientError, KeyError, StopIteration) as error:
        print(f"Staging stack is not discoverable from CodePipeline yet: {error}")

    # The endpoint name is deterministic in endpoint-config-template.yml. If the staging
    # stack already exists, prefer its physical Endpoint ID as the source of truth.
    endpoint_name = f"{PROJECT_NAME}-staging"
    deadline = time.time() + WAIT_FOR_STAGING_ENDPOINT_SECONDS
    while True:
        if staging_stack_name:
            try:
                stack_resources = cfn.describe_stack_resources(StackName=staging_stack_name)["StackResources"]
                endpoint_resource = next(
                    (item for item in stack_resources if item["LogicalResourceId"] == "Endpoint"),
                    None,
                )
                if endpoint_resource and endpoint_resource.get("PhysicalResourceId"):
                    physical_id = endpoint_resource["PhysicalResourceId"]
                    # SageMaker resources may be reported as either a name or a full ARN.
                    endpoint_name = physical_id.rsplit("/", 1)[-1]
            except ClientError:
                pass

        try:
            endpoint_status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
            break
        except ClientError as error:
            if error.response["Error"]["Code"] != "ValidationException" or time.time() >= deadline:
                break
            time.sleep(15)

    studio_domain_id = (
        project_details.get("CreatedBy", {}).get("DomainId")
        or DOMAIN_ID
        or globals().get("domain_id")
    )
    if not studio_domain_id:
        print(f"Staging endpoint: {endpoint_name} ({endpoint_status})")
        print("Set DOMAIN_ID to render its SageMaker Studio link.")
    else:
        endpoint_url = (
            f"https://studio-{studio_domain_id}.studio.{region}.sagemaker.aws/"
            f"inference-experience/endpoints/details/{quote(endpoint_name, safe='')}"
        )
        display(HTML(
            f'<p><b>Staging endpoint:</b> <code>{escape(endpoint_name)}</code> '
            f'(<b>{escape(endpoint_status)}</b>)<br>'
            f'<a target="_blank" rel="noopener noreferrer" href="{escape(endpoint_url)}">'
            "Open the staging endpoint in SageMaker Studio</a></p>"
        ))


![](img/staging-endpoint.png)

## Smoke-test the staging endpoint

Send one held-out feature row using the same `text/csv` contract validated in notebook 04. The shared helper checks schema, endpoint status, data capture, HTTP/MIME type, response cardinality and the probability range. A unique `InferenceId` makes each request traceable in captured data.

In [ ]:
import csv
import math
from urllib.parse import urlparse

runtime = session.client("sagemaker-runtime", region_name=region)


def load_smoke_payload():
    """Return one validated held-out CSV row and its source."""
    local_test_file = Path("tmp/test_x.csv")
    if local_test_file.is_file():
        with local_test_file.open(encoding="utf-8") as handle:
            payload = next((line.strip() for line in handle if line.strip()), None)
        payload_source = str(local_test_file)
    else:
        stored_test_s3_url = globals().get("test_s3_url")
        if not stored_test_s3_url:
            raise RuntimeError(
                "No test data found. Run notebook 01 or 03 so test_s3_url is stored, "
                "or run notebook 04 to create tmp/test_x.csv."
            )
        parsed = urlparse(stored_test_s3_url)
        if parsed.scheme != "s3" or not parsed.netloc:
            raise ValueError(f"Expected an S3 URL in test_s3_url, got {stored_test_s3_url!r}")
        test_key = f"{parsed.path.strip('/')}/test_x.csv"
        sample_object = s3.get_object(
            Bucket=parsed.netloc, Key=test_key, Range="bytes=0-65535"
        )
        sample_text = sample_object["Body"].read().decode("utf-8")
        payload = next((line.strip() for line in sample_text.splitlines() if line.strip()), None)
        payload_source = f"s3://{parsed.netloc}/{test_key}"

    if not payload:
        raise ValueError(f"No non-empty test row found in {payload_source}")
    fields = next(csv.reader([payload]))
    try:
        numeric_fields = [float(value) for value in fields]
    except ValueError as error:
        raise ValueError(f"Test payload contains a non-numeric feature: {error}") from error

    expected_feature_names = globals().get("feature_names")
    if not expected_feature_names:
        raise RuntimeError("feature_names is unavailable; run notebook 01 or 03 first.")
    if len(numeric_fields) != len(expected_feature_names):
        raise ValueError(
            f"Payload has {len(numeric_fields)} features, expected {len(expected_feature_names)}. "
            f"The test file may be stale: {payload_source}"
        )
    if not all(math.isfinite(value) for value in numeric_fields):
        raise ValueError("Test payload contains NaN or an infinite feature value.")
    return payload, payload_source, len(numeric_fields)


def wait_for_endpoint(endpoint_to_wait_for, timeout_seconds=900):
    """Wait for one endpoint to become InService; fail on terminal status."""
    deadline = time.time() + timeout_seconds
    last_status = "NotFound"
    while True:
        try:
            description = sm.describe_endpoint(EndpointName=endpoint_to_wait_for)
            last_status = description["EndpointStatus"]
        except ClientError as error:
            if error.response["Error"]["Code"] != "ValidationException":
                raise
            last_status = "NotFound"

        if last_status == "InService":
            return description
        if last_status in {"Failed", "OutOfService", "UpdateRollbackFailed", "RollbackFailed"}:
            raise RuntimeError(
                f"Endpoint {endpoint_to_wait_for} entered terminal status {last_status}."
            )
        if time.time() >= deadline:
            raise TimeoutError(
                f"Endpoint {endpoint_to_wait_for} did not become InService within "
                f"{timeout_seconds} seconds; last status: {last_status}."
            )
        print(f"  {time.strftime('%H:%M:%S')} | {endpoint_to_wait_for}={last_status}")
        time.sleep(20)


def smoke_test_endpoint(endpoint_to_test, label):
    """Invoke one row and require one finite probability in [0, 1]."""
    endpoint_description = wait_for_endpoint(endpoint_to_test)
    config = sm.describe_endpoint_config(
        EndpointConfigName=endpoint_description["EndpointConfigName"]
    )
    if not config.get("DataCaptureConfig", {}).get("EnableCapture"):
        raise AssertionError(f"Data capture is not enabled for {endpoint_to_test}")

    payload, payload_source, feature_count = load_smoke_payload()
    inference_id = f"{label.lower()}-smoke-{time.time_ns()}"
    started = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_to_test,
        ContentType="text/csv",
        Accept="text/csv",
        Body=payload,
        InferenceId=inference_id,
    )
    latency_ms = (time.perf_counter() - started) * 1000

    http_status = response["ResponseMetadata"]["HTTPStatusCode"]
    content_type = response.get("ContentType", "").split(";", 1)[0].strip().lower()
    body = response["Body"].read().decode("utf-8").strip()
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    if http_status != 200:
        raise AssertionError(f"Expected HTTP 200, got {http_status}")
    if content_type != "text/csv":
        raise AssertionError(f"Expected text/csv response, got {content_type!r}")
    if len(lines) != 1:
        raise AssertionError(f"Expected one prediction, got {len(lines)}: {body!r}")
    try:
        prediction = float(lines[0])
    except ValueError as error:
        raise AssertionError(f"Prediction is not numeric: {lines[0]!r}") from error
    if not math.isfinite(prediction) or not 0.0 <= prediction <= 1.0:
        raise AssertionError(f"Expected a finite probability in [0, 1], got {prediction}")

    result = {
        "endpoint": endpoint_to_test,
        "status": endpoint_description["EndpointStatus"],
        "feature_count": feature_count,
        "payload_source": payload_source,
        "http_status": http_status,
        "content_type": content_type,
        "prediction": prediction,
        "latency_ms": round(latency_ms, 1),
        "inference_id": inference_id,
    }
    print(json.dumps(result, indent=2))
    display(HTML(
        f'<div style="border:2px solid #1b5e20;border-radius:6px;padding:10px;'
        f'background:#e8f5e9"><b>{escape(label)} endpoint smoke test passed.</b></div>'
    ))
    return result

In [ ]:
staging_smoke_result = None
if not globals().get("endpoint_name"):
    print("Skipped — no staging endpoint has been discovered.")
else:
    staging_smoke_result = smoke_test_endpoint(endpoint_name, "Staging")

## Guarded production approval

The staging stack and its CodePipeline test must succeed before production can be approved. Approval creates a second billable real-time endpoint. Inspect the staging smoke-test result and the pipeline link above, then set `APPROVE_PRODUCTION = True` in the settings cell and run the next cell.

The manual approval is the third action of the `DeployStaging` stage, after the CloudFormation deployment of the staging endpoint and the `TestStaging` CodeBuild run. The staging endpoint being `InService` does not mean the pipeline has reached it yet: the test still runs for a minute or two. The cell therefore follows the current pipeline execution until `ApproveDeployment` is waiting for a decision (up to `APPROVAL_WAIT_SECONDS`), and stops with the failing action if the execution fails on the way. It approves only the currently waiting action and cannot silently approve a different execution.

In [ ]:
APPROVAL_WAIT_SECONDS = 900

APPROVAL_STAGE, APPROVAL_ACTION = "DeployStaging", "ApproveDeployment"
FAILED_STATUSES = {"Failed", "Stopped", "Stopping", "Abandoned", "Superseded"}

def latest_pipeline_execution():
    summaries = codepipeline.list_pipeline_executions(
        pipelineName=pipeline_name, maxResults=1
    )["pipelineExecutionSummaries"]
    return summaries[0] if summaries else None

def pipeline_progress():
    """Current (stage, action, action latestExecution) triples, in pipeline order."""
    state = codepipeline.get_pipeline_state(name=pipeline_name)
    return [
        (stage["stageName"], action["actionName"], action.get("latestExecution", {}))
        for stage in state["stageStates"]
        for action in stage["actionStates"]
    ]

def describe_failure(progress):
    for stage, action, execution in progress:
        if execution.get("status") == "Failed":
            details = execution.get("errorDetails", {}).get("message") or execution.get("summary", "")
            url = execution.get("externalExecutionUrl", "")
            return f"{stage}/{action} failed: {details}" + (f"\n  logs: {url}" if url else "")
    return "the pipeline execution did not reach the approval action"

approval_status = "Unavailable"
approval_token = None

if not globals().get("pipeline_name"):
    print("No deployment pipeline is available for approval.")
    if APPROVE_PRODUCTION:
        raise RuntimeError("Cannot approve production without a deployment pipeline.")
else:
    execution = latest_pipeline_execution()
    if execution is None:
        raise RuntimeError(
            f"Pipeline {pipeline_name} has never run. Its EventBridge rule only reacts to Model "
            "Package state changes that happen after the Project exists, and the source rule only "
            "to commits in the generated repository. Approve a model package version in the group "
            "(notebook 03 or the Studio Model Registry) or start the pipeline manually with "
            f"codepipeline.start_pipeline_execution(name={pipeline_name!r}), then re-run this cell."
        )
    print(f"Pipeline execution {execution['pipelineExecutionId']}: {execution['status']}")

    deadline = time.time() + APPROVAL_WAIT_SECONDS
    while True:
        progress = pipeline_progress()
        approval_execution = next(
            exec_ for stage, action, exec_ in progress
            if stage == APPROVAL_STAGE and action == APPROVAL_ACTION
        )
        approval_status = approval_execution.get("status", "NotStarted")
        approval_token = approval_execution.get("token")
        if approval_status in {"InProgress", "Succeeded"}:
            break
        execution = latest_pipeline_execution()
        if execution["status"] in FAILED_STATUSES:
            raise RuntimeError(
                f"Pipeline execution {execution['pipelineExecutionId']} is {execution['status']}; "
                f"{describe_failure(progress)}.\nFix the cause (a failed build usually means the "
                "model package group has no Approved version yet) and re-run this cell once a new "
                "execution is in progress."
            )
        if time.time() >= deadline:
            raise TimeoutError(
                f"{APPROVAL_STAGE}/{APPROVAL_ACTION} was not reached within {APPROVAL_WAIT_SECONDS}s; "
                "the execution is still in progress. Re-run this cell to keep waiting."
            )
        running = [f"{s}/{a}" for s, a, e in progress if e.get("status") == "InProgress"]
        print(f"  {time.strftime('%H:%M:%S')} | waiting for the approval action; "
              f"in progress: {', '.join(running) or '(between actions)'}")
        time.sleep(20)

    print(f"Production approval status: {approval_status}")
    if not APPROVE_PRODUCTION:
        print("PREVIEW ONLY — set APPROVE_PRODUCTION = True after reviewing staging.")
    elif approval_status == "Succeeded":
        print("This pipeline execution is already approved; no action taken.")
    elif not approval_token:
        raise RuntimeError(
            f"Approval action is InProgress but returned no token; status={approval_status}. "
            "Re-run this cell."
        )
    else:
        approval_result = codepipeline.put_approval_result(
            pipelineName=pipeline_name,
            stageName=APPROVAL_STAGE,
            actionName=APPROVAL_ACTION,
            result={
                "summary": "Staging endpoint and real CSV inference smoke test reviewed in notebook 05.",
                "status": "Approved",
            },
            token=approval_token,
        )
        approval_status = "Approved"
        print(f"Production approved: {approval_result['approvedAt']}")

## Production endpoint link and smoke test

After approval, optionally set `WAIT_FOR_PRODUCTION_ENDPOINT = True` to wait up to 15 minutes for the production endpoint. The Studio link is shown immediately and becomes active when the endpoint appears. Set `TEST_PRODUCTION_ENDPOINT = True` only after production is `InService`; it reuses the exact staging smoke-test contract.

In [ ]:
production_endpoint_name = f"{PROJECT_NAME}-prod"
production_endpoint_status = "NotFound"
production_stack_name = None
production_endpoint_url = None

if not globals().get("pipeline_name") or project_details is None:
    print("No deployment pipeline is available; production has not been deployed.")
else:
    try:
        pipeline_definition = codepipeline.get_pipeline(name=pipeline_name)["pipeline"]
        production_action = next(
            action
            for stage in pipeline_definition["stages"] if stage["name"] == "DeployProd"
            for action in stage["actions"] if action["name"] == "DeployResourcesProd"
        )
        production_stack_name = production_action["configuration"]["StackName"]
    except (ClientError, KeyError, StopIteration) as error:
        print(f"Production stack is not discoverable from CodePipeline yet: {error}")

    wait_deadline = time.time() + 900
    while True:
        if production_stack_name:
            try:
                resources = cfn.describe_stack_resources(
                    StackName=production_stack_name
                )["StackResources"]
                endpoint_resource = next(
                    (item for item in resources if item["LogicalResourceId"] == "Endpoint"),
                    None,
                )
                if endpoint_resource and endpoint_resource.get("PhysicalResourceId"):
                    production_endpoint_name = endpoint_resource["PhysicalResourceId"].rsplit("/", 1)[-1]
            except ClientError:
                pass
        try:
            production_endpoint_status = sm.describe_endpoint(
                EndpointName=production_endpoint_name
            )["EndpointStatus"]
            break
        except ClientError as error:
            if error.response["Error"]["Code"] != "ValidationException":
                raise
            if not WAIT_FOR_PRODUCTION_ENDPOINT or time.time() >= wait_deadline:
                break
            print(f"  {time.strftime('%H:%M:%S')} | production endpoint not visible yet")
            time.sleep(20)

    studio_domain_id = (
        project_details.get("CreatedBy", {}).get("DomainId")
        or DOMAIN_ID
        or globals().get("domain_id")
    )
    if studio_domain_id:
        production_endpoint_url = (
            f"https://studio-{studio_domain_id}.studio.{region}.sagemaker.aws/"
            f"inference-experience/endpoints/details/{quote(production_endpoint_name, safe='')}"
        )
        display(HTML(
            f'<p><b>Production endpoint:</b> <code>{escape(production_endpoint_name)}</code> '
            f'(<b>{escape(production_endpoint_status)}</b>)<br>'
            f'<a target="_blank" rel="noopener noreferrer" href="{escape(production_endpoint_url)}">'
            "Open the production endpoint in SageMaker Studio</a></p>"
        ))
    else:
        print(f"Production endpoint: {production_endpoint_name} ({production_endpoint_status})")

In [ ]:
production_smoke_result = None
if not TEST_PRODUCTION_ENDPOINT:
    print("Skipped — set TEST_PRODUCTION_ENDPOINT = True after production is InService.")
else:
    production_smoke_result = smoke_test_endpoint(production_endpoint_name, "Production")

## Guarded cleanup

Cleanup is destructive and disabled by default. It first abandons any active pipeline execution, then deletes the separately managed production and staging endpoint stacks, and finally deletes the SageMaker Project/toolchain stack. The Project artifact bucket has `DeletionPolicy: Retain`; deleting it is a second independent opt-in. The publication bucket containing the reusable template and seed ZIP is never deleted.

In [ ]:
def stack_exists(stack_name):
    try:
        cfn.describe_stacks(StackName=stack_name)
        return True
    except ClientError as error:
        if "does not exist" in error.response["Error"]["Message"]:
            return False
        raise


def empty_artifact_bucket(bucket_name):
    # Delete all versions and delete markers when versioning is enabled.
    version_paginator = s3.get_paginator("list_object_versions")
    for page in version_paginator.paginate(Bucket=bucket_name):
        objects = [
            {"Key": item["Key"], "VersionId": item["VersionId"]}
            for collection in (page.get("Versions", []), page.get("DeleteMarkers", []))
            for item in collection
        ]
        for start in range(0, len(objects), 1000):
            s3.delete_objects(
                Bucket=bucket_name,
                Delete={"Objects": objects[start:start + 1000], "Quiet": True},
            )
    # Also cover an unversioned bucket.
    object_paginator = s3.get_paginator("list_objects_v2")
    for page in object_paginator.paginate(Bucket=bucket_name):
        objects = [{"Key": item["Key"]} for item in page.get("Contents", [])]
        for start in range(0, len(objects), 1000):
            s3.delete_objects(
                Bucket=bucket_name,
                Delete={"Objects": objects[start:start + 1000], "Quiet": True},
            )


if project_details is None or not globals().get("project_id"):
    print("No Project exists; there are no deployment resources to clean up.")
    if CLEAN_UP_DEPLOYMENT:
        raise RuntimeError("Cannot clean up without a resolved Project ID.")
else:
    artifact_bucket = f"sagemaker-project-{project_id}"
    staging_stack_name = f"sagemaker-{PROJECT_NAME}-{project_id}-deploy-staging"
    production_stack_name = f"sagemaker-{PROJECT_NAME}-{project_id}-deploy-prod"
    print("Cleanup targets:")
    print(f"  pipeline:    {pipeline_name}")
    print(f"  prod stack:  {production_stack_name}")
    print(f"  stage stack: {staging_stack_name}")
    print(f"  project:     {PROJECT_NAME}")
    print(f"  retained S3: {artifact_bucket} (delete={DELETE_RETAINED_ARTIFACT_BUCKET})")

    if not CLEAN_UP_DEPLOYMENT:
        print("PREVIEW ONLY — set CLEAN_UP_DEPLOYMENT = True to delete deployment resources.")
    else:
        executions = codepipeline.list_pipeline_executions(
            pipelineName=pipeline_name, maxResults=10
        )["pipelineExecutionSummaries"]
        active_execution_ids = []
        for execution in executions:
            if execution["status"] == "InProgress":
                execution_id = execution["pipelineExecutionId"]
                codepipeline.stop_pipeline_execution(
                    pipelineName=pipeline_name,
                    pipelineExecutionId=execution_id,
                    abandon=True,
                    reason="Notebook 05 guarded cleanup",
                )
                active_execution_ids.append(execution_id)
                print(f"Stopping pipeline execution {execution_id}")
        for execution_id in active_execution_ids:
            deadline = time.time() + 300
            while True:
                status = codepipeline.get_pipeline_execution(
                    pipelineName=pipeline_name,
                    pipelineExecutionId=execution_id,
                )["pipelineExecution"]["status"]
                if status != "InProgress":
                    print(f"Pipeline execution {execution_id}: {status}")
                    break
                if time.time() >= deadline:
                    raise TimeoutError(f"Pipeline execution {execution_id} did not stop")
                time.sleep(10)

        for stack_name in [production_stack_name, staging_stack_name]:
            if stack_exists(stack_name):
                print(f"Deleting {stack_name}")
                cfn.delete_stack(StackName=stack_name)
                cfn.get_waiter("stack_delete_complete").wait(
                    StackName=stack_name,
                    WaiterConfig={"Delay": 20, "MaxAttempts": 90},
                )
            else:
                print(f"Stack does not exist: {stack_name}")

        try:
            current_project = sm.describe_project(ProjectName=PROJECT_NAME)
        except ClientError as error:
            if error.response["Error"]["Code"] != "ValidationException":
                raise
            current_project = None
        if current_project and current_project["ProjectStatus"] != "DeleteCompleted":
            sm.delete_project(ProjectName=PROJECT_NAME)
            deadline = time.time() + 1800
            while True:
                try:
                    current_project = sm.describe_project(ProjectName=PROJECT_NAME)
                    status = current_project["ProjectStatus"]
                except ClientError as error:
                    if error.response["Error"]["Code"] != "ValidationException":
                        raise
                    status = "Deleted"
                print(f"  {time.strftime('%H:%M:%S')} | project={status}")
                if status in {"DeleteCompleted", "Deleted"}:
                    break
                if status == "DeleteFailed":
                    raise RuntimeError(
                        current_project.get("ProjectStatusMessage", "Project deletion failed")
                    )
                if time.time() >= deadline:
                    raise TimeoutError("Project deletion did not complete within 30 minutes")
                time.sleep(20)

        if DELETE_RETAINED_ARTIFACT_BUCKET:
            try:
                empty_artifact_bucket(artifact_bucket)
                s3.delete_bucket(Bucket=artifact_bucket)
                print(f"Deleted retained bucket {artifact_bucket}")
            except ClientError as error:
                if error.response["Error"]["Code"] == "NoSuchBucket":
                    print(f"Bucket already absent: {artifact_bucket}")
                else:
                    raise
        else:
            print(f"Retained bucket left in place: s3://{artifact_bucket}")

## What to adapt next

The generated CodeCommit repository is initialized once from the ZIP published by this notebook. The local seed now contains a real 64-feature inference fixture and pipeline gate. To apply later local seed changes, republish and create a new Project; for an existing Project, commit the corresponding files directly to its generated CodeCommit repository.

Before adapting this reference for production:

1. Replace `test/payload.csv` and `--expected-feature-count 64` with a fixture/schema owned by your model contract.
2. Review instance types, counts and capture percentages in both stage config files.
3. Decide whether `build.py` should require the workshop's `production/active` lifecycle in addition to `Approved`.
4. Move to CodeConnections and your organization's supported Git provider.
5. Keep production approval and cleanup flags false until the corresponding action is explicitly intended.